### Requirements 

#### .env file

```bash
MILVUS_URL = # your milvus server url
MILVUS_TOKEN = # your milvus access token
MILVUS_COLLECTION_NAME = # milvus collection name
```

In [1]:
import os
import sys 
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
load_dotenv(override=True)
ROOT = Path.cwd().parent.parent.parent.resolve()
current_dir = Path.cwd()
project_root = current_dir.resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Added to path: {project_root}")

Added to path: /users/oshan/Dev/financial-document-based-agent-system


### Milvus server Info

In [2]:
import pymilvus
import os
from urllib.parse import urlparse

In [3]:
milvus_url = os.environ.get("MILVUS_URL")
milvus_token = os.environ.get("MILVUS_TOKEN")
milvus_collection_name = os.environ.get("MILVUS_COLLECTION_NAME")

if not milvus_url:
    raise RuntimeError("MILVUS_URL environment variable not set")

# parse host/port if needed
u = urlparse(milvus_url if "://" in milvus_url else f"//{milvus_url}")
host = u.hostname or milvus_url
port = u.port or 19530

# try connecting (try uri first, then host/port with optional token as password)
errors = []
try:
    pymilvus.connections.connect(uri=milvus_url)
except Exception as e:
    errors.append(e)
    try:
        if milvus_token:
            pymilvus.connections.connect(host=host, port=str(port), password=milvus_token)
        else:
            pymilvus.connections.connect(host=host, port=str(port))
    except Exception as e2:
        errors.append(e2)
        raise RuntimeError("Failed to connect to Milvus", errors)

print("Connected to:", milvus_url)
print("Collections:", pymilvus.utility.list_collections())

Connected to: http://localhost:19530
Collections: ['agent_collection_3eb2d5d9309b4a37b581cabbf24a96d1', 'agent_collection_c98c9ef0b6744f79be365169eca3791d', 'agent_collection_e6aab4df2bbe47868afd76ae750703a5', 'agent_collection_02a3523a75764604b9cef115ce973b75', 'test_collection', 'test_collection_2', 'agent_collection_c1dfdfce36ca41c6ad3654f6fca875ba', 'agent_collection_540bd51003194a86ad29e903cd246d0c', 'agent_collection_e1c89aee5e7b4d9090a775dec10f6a2d', 'agent_collection_7506792f8d704bcda0ec95f459d7c2b1', 'agent_collection_bd880a40088e4a779dfa20ae1e5fe1fa', 'agent_collection_b990d9c0e8b045238a094cefd5fcdceb', 'agent_collection_4f0c271900364d62b23491c9a1f0deb4', 'agent_collection_a2063496ad59407680002b6f3e680956', 'agent_collection_290baf6466b24823a45d0a1bd9b11ae6', 'agent_collection_6d7fc18e3cec45cda04722dec7603b4b', 'agent_collection_ebfaf66bcc6442a4a7b9920b6eeef9e0', 'agent_collection_b487e0efdb9541718034431bb0d8ae86', 'agent_collection_5c1cdc2edf92487883eb8e2c5efffc1e', 'xml_doc

In [4]:
milvus_url = os.environ.get("MILVUS_URL")
milvus_token = os.environ.get("MILVUS_TOKEN")
milvus_collection_name = os.environ.get("MILVUS_COLLECTION_NAME")

In [5]:
milvus_collection_name in pymilvus.utility.list_collections()

True

### Class info

In [6]:
from src.cgcore.vectordb.milvus import MilvusDB
from src.cgcore.configs.vectordb.milvus import MilvusConfig

In [7]:
config = MilvusConfig(**{
    "collection_name": milvus_collection_name,
    "dimensions": 1536,
})

milvus_db = MilvusDB(config)

AsyncMilvusClient initialized.


In [8]:
config.collection_name

'financial_documents_backend_test'

#### Insert text

In [9]:
question = "What is Milvus?"
embedding = [0.0] * 1536  # Dummy embedding for testing
content = "Milvus is an open-source vector database."
docid = "test_paper_1"

milvus_db.insert([
    {
        "content": content, 
        "vector": embedding, 
        "doc_id": docid, 
        "meta_data": {"source": "test_source"}
    },
    {
        "content": "Milvus supports efficient similarity search.", 
        "vector": embedding, 
        "doc_id": "test_paper_2", 
        "meta_data": {"source": "test_source"}
    }
])

<coroutine object MilvusDB.insert at 0xf16569e0eb20>

#### Retrived text

In [10]:
milvus_db.vector_search(embedding, top_k=2)

<coroutine object MilvusDB.vector_search at 0xf16519e23a00>

### Drop collection if necessary

In [11]:
if pymilvus.utility.has_collection(config.collection_name):
    pymilvus.utility.drop_collection(config.collection_name)
    print(f"Dropped collection: {config.collection_name}")
else:
    print(f"Collection not found: {config.collection_name}")
print("Collections now:", pymilvus.utility.list_collections())

Dropped collection: financial_documents_backend_test
Collections now: ['xml_documents_qwen8', 'embedchain_store', 'agent_collection_e3634f7638c6467a92d67bf4e44b7c20', 'agent_collection_2d68d6d879ca4405b03c1cb44fa1a6db', 'agent_collection_5792e54f3cdc4dfe888102a15349caa2', 'agent_collection_22f60bc5dfa14724833fadf7034ce961', 'agent_collection_54dc03653ea341a589e521699ac6fbe5', 'xml_documents', 'agent_collection_5a86b65a493646a3be2b7330bd420657', 'agent_collection_5232b146d3ae48d999b9c9abe8d0ae30', 'financial_documents_test', 'my_rag_collection', 'agent_collection_7ecffa4ffcd3426fb5a41283caa7eb19', 'agent_collection_37d59e7e92484a1ea6b20371545fbb8f', 'agent_collection_994de10f738a4063934e605bd280b911', 'agent_collection_27ad2a0e014948e89b95ab5145f5aa5e', 'xml_documentsv4', 'agent_collection_8ace8e5726fe447a8294677261ed524c', 'agent_collection_c8fa94a8875a4ac7ac394e1a138f071b', 'agent_collection_2db8c9ab6a86420a982b7a2ec7c5e09e', 'agent_collection_dd8ace4434454b368b0a27e835c1e8bb', 'agent